# Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
GPU Name: NVIDIA L4


# Install **pytorch-tabnet, optuna**

In [ ]:
!pip install pytorch-tabnet optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 45.0 MB/s eta 0:00:00


# Important Libraries

In [ ]:
import optuna
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from pytorch_tabnet.metrics import Metric
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric
import joblib
import os
import gc

# Sentinel-2

## Data Loading

In [ ]:
save_dir = "/content/drive/MyDrive/Thesis/Trained_models/"
file_path = "/content/drive/MyDrive/Thesis/Train_data/s2_with_veg_data.csv"

print("Loading data...")
df = pd.read_csv(file_path)
print("Data is loaded")


print("\nData preparation...")
df.drop(columns=["height", "stock_per_ha", "basal_area", "poly_id", "S2_Date"], axis=1, inplace=True)

df1 = df.loc[df["age"] != 5]

# Prepare X and y
y = (df1["age"] - 1).values
X = df1.drop(columns="age", axis=1).values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

print(f"\nDatasets are read! X_train shape: {X_train.shape}, X_test shape: {X_test.shape}, X_val shape: {X_val.shape}")

Loading data...
Data is loaded

Data preparation...

Datasets are read! X_train shape: (284286, 52), X_test shape: (50169, 52), X_val shape: (59022, 52)


### Handle Imbalanced Class

In [ ]:
print("Calculating Smoothed Class Weights...")

unique_classes = np.unique(y_train)

standard_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)

smoothed_class_weights = np.sqrt(standard_class_weights)

smoothed_weight_dict = {
    int(class_label): float(weight)
    for class_label, weight in zip(unique_classes, smoothed_class_weights)
}

print("Smoothed Weights Dictionary:", smoothed_weight_dict)

Calculating Smoothed Class Weights...
Smoothed Weights Dictionary: {0: 1.6860445694430484, 1: 0.6375345794450832, 2: 1.3163812181075647, 3: 1.279507522681642}


##Bayesian Optimization

In [ ]:
# PyTorch and NVIDIA GPUs require float32 to run at maximum speed!
X_train_np = X_train.values.astype(np.float32) if isinstance(X_train, pd.DataFrame) else X_train.astype(np.float32)
y_train_np = y_train.values.astype(int) if isinstance(y_train, pd.Series) else y_train.astype(int)

X_val_np = X_val.values.astype(np.float32) if isinstance(X_val, pd.DataFrame) else X_val.astype(np.float32)
y_val_np = y_val.values.astype(int) if isinstance(y_val, pd.Series) else y_val.astype(int)

X_test_np = X_test.values.astype(np.float32) if isinstance(X_test, pd.DataFrame) else X_test.astype(np.float32)
y_test_np = y_test.values.astype(int) if isinstance(y_test, pd.Series) else y_test.astype(int)

In [ ]:
def objective(trial):
    """
    Optuna will run this function multiple times.
    In each trial, it picks a smart combination of parameters to test.
    """

    # Define the Hyperparameter Search Space
    n_d = trial.suggest_int('n_d', 8, 64, step=8)
    n_a = n_d # It is highly recommended to keep n_d and n_a equal
    n_steps = trial.suggest_int('n_steps', 3, 8)
    gamma = trial.suggest_float('gamma', 1.0, 2.0)
    lambda_sparse = trial.suggest_float('lambda_sparse', 1e-5, 1e-2, log=True)
    learning_rate = trial.suggest_float('lr', 1e-3, 1e-1, log=True)

    # Initialize TabNet with the suggested parameters
    clf = TabNetClassifier(
        n_d=n_d,
        n_a=n_a,
        n_steps=n_steps,
        gamma=gamma,
        lambda_sparse=lambda_sparse,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=learning_rate),
        scheduler_params={"step_size":10, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', # "entmax" yields better, sparser attention than "sparsemax"
        device_name='cuda',
        verbose=0
    )

    # Train the Model
    clf.fit(
        X_train=X_train_np, y_train=y_train_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_name=['valid'],
        eval_metric=['logloss'], # Optimize logloss internally
        max_epochs=50,           # Keep relatively short for tuning
        patience=10,             # Early stopping if validation doesn't improve
        batch_size=1024,
        virtual_batch_size=128,
        weights=smoothed_weight_dict                # Fixes the imbalanced age classes
    )

    # 4. Evaluate and return the score Optuna needs to maximize
    preds = clf.predict(X_val_np)
    f1 = f1_score(y_val_np, preds, average='macro')

    return f1

# RUN THE BAYESIAN SEARCH
print("\nStarting Optuna Bayesian Optimization for TabNet...")

# Create the study. We want to MAXIMIZE the f1_macro score.
study = optuna.create_study(direction='maximize')


study.optimize(objective, n_trials=50)

print("\nBest TabNet Parameters Found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# TRAIN THE FINAL TABNET MODEL
print("\nTraining Final TabNet Model with Best Parameters...")

# Extract the winning parameters
best_params = study.best_params
best_n_d = best_params['n_d']

final_tabnet = TabNetClassifier(
    n_d=best_n_d,
    n_a=best_n_d,
    n_steps=best_params['n_steps'],
    gamma=best_params['gamma'],
    lambda_sparse=best_params['lambda_sparse'],
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=best_params['lr']),
    mask_type='entmax',
    device_name='cuda',
    verbose=1
)

final_tabnet.fit(
    X_train=X_train_np, y_train=y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    eval_name=['valid'],
    eval_metric=['logloss'],
    max_epochs=150, # Let the final model train much longer
    patience=20,
    batch_size=1024, virtual_batch_size=128,
    weights=smoothed_weight_dict
)

[I 2026-04-07 16:24:30,444] A new study created in memory with name: no-name-8dc8421f-405e-442a-a2ec-d7d43df7bc8c



Starting Optuna Bayesian Optimization for TabNet...
Stop training because you reached max_epochs = 50 with best_epoch = 42 and best_valid_logloss = 0.93618


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-07 16:43:18,191] Trial 0 finished with value: 0.4471617300897657 and parameters: {'n_d': 56, 'n_steps': 6, 'gamma': 1.5448528218136293, 'lambda_sparse': 0.0016162863251257282, 'lr': 0.0023917453340365144}. Best is trial 0 with value: 0.4471617300897657.



Early stopping occurred at epoch 36 with best_epoch = 26 and best_valid_logloss = 0.93196


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-07 16:53:30,841] Trial 1 finished with value: 0.4398466794850049 and parameters: {'n_d': 8, 'n_steps': 4, 'gamma': 1.1031770752327443, 'lambda_sparse': 0.0005321649144906653, 'lr': 0.022839129499238258}. Best is trial 0 with value: 0.4471617300897657.


...

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-08 06:17:41,201] Trial 48 finished with value: 0.4783063763254781 and parameters: {'n_d': 24, 'n_steps': 8, 'gamma': 1.750322226451336, 'lambda_sparse': 0.0001139942191047631, 'lr': 0.011543523832224574}. Best is trial 13 with value: 0.5050888345548867.



Early stopping occurred at epoch 49 with best_epoch = 39 and best_valid_logloss = 0.93188


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-08 06:34:28,622] Trial 49 finished with value: 0.4549009500808086 and parameters: {'n_d': 8, 'n_steps': 5, 'gamma': 1.1763178573567732, 'lambda_sparse': 0.0031079377354161636, 'lr': 0.006527051676027586}. Best is trial 13 with value: 0.5050888345548867.
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")



Best TabNet Parameters Found:
  n_d: 64
  n_steps: 7
  gamma: 1.6627277981428936
  lambda_sparse: 0.0024679054940156702
  lr: 0.006594489269960494

Training Final TabNet Model with Best Parameters...
epoch 0  | loss: 1.48961 | valid_logloss: 1.10717 |  0:00:25s
epoch 1  | loss: 1.21675 | valid_logloss: 1.09827 |  0:00:51s
epoch 2  | loss: 1.2075  | valid_logloss: 1.066   |  0:01:16s
epoch 3  | loss: 1.19833 | valid_logloss: 1.04694 |  0:01:42s
epoch 4  | loss: 1.18038 | valid_logloss: 1.06332 |  0:02:07s
epoch 5  | loss: 1.17318 | valid_logloss: 1.04921 |  0:02:33s
epoch 6  | loss: 1.17458 | valid_logloss: 1.04985 |  0:02:58s
epoch 7  | loss: 1.16192 | valid_logloss: 1.0481  |  0:03:24s
epoch 8  | loss: 1.14791 | valid_logloss: 1.01123 |  0:03:49s
epoch 9  | loss: 1.14161 | valid_logloss: 1.03513 |  0:04:15s
epoch 10 | loss: 1.12916 | valid_logloss: 1.03124 |  0:04:41s
epoch 11 | loss: 1.12599 | valid_logloss: 1.00836 |  0:05:06s
epoch 12 | loss: 1.11591 | valid_logloss: 0.97426 |  0:

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


**Best Params**

In [ ]:
# Best TabNet Parameters Found:
#   n_d: 64
#   n_steps: 7
#   gamma: 1.6627277981428936
#   lambda_sparse: 0.0024679054940156702
#   lr: 0.006594489269960494

#Alpha Earth Foundation + Sentinel-2

## Data Loading

In [ ]:
save_dir = "/content/drive/MyDrive/Thesis/Trained_models/"
file_path = "/content/drive/MyDrive/Thesis/Train_data/s2_aef_with_veg_data.csv"

print("Loading data...")
df = pd.read_csv(file_path)
print("Data is loaded")


print("\nData preparation...")

df.drop(columns=["height", "stock_per_ha", "basal_area", "poly_id", "S2_Date"], axis=1, inplace=True)

df1 = df.loc[df["age"] != 5]

df1.dropna(inplace=True)

y = df1["age"] - 1
X = df1.drop(columns="age", axis=1)

# Split data into Train, Validation, and Test sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

print(f"\nDatasets are read! X_train shape: {X_train.shape}, X_test shape: {X_test.shape}, X_val shape: {X_val.shape}")

Loading data...
Data is loaded

Data preparation...


/tmp/ipykernel_23756/3787584650.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.dropna(inplace=True)



Datasets are read! X_train shape: (284117, 308), X_test shape: (50139, 308), X_val shape: (58987, 308)


### Handle Imbalanced Class

In [ ]:
print("Calculating Smoothed Class Weights...")

unique_classes = np.unique(y_train)

standard_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)

smoothed_class_weights = np.sqrt(standard_class_weights)

smoothed_weight_dict = {
    int(class_label): float(weight)
    for class_label, weight in zip(unique_classes, smoothed_class_weights)
}

print("Smoothed Weights Dictionary:", smoothed_weight_dict)

Calculating Smoothed Class Weights...
Smoothed Weights Dictionary: {0: 1.6887210001812631, 1: 0.637314073657483, 2: 1.3186288824923265, 3: 1.278067734378095}


##Bayesian Optimization

In [ ]:
# PyTorch and NVIDIA GPUs require float32 to run at maximum speed!
X_train_np = X_train.values.astype(np.float32) if isinstance(X_train, pd.DataFrame) else X_train.astype(np.float32)
y_train_np = y_train.values.astype(int) if isinstance(y_train, pd.Series) else y_train.astype(int)

X_val_np = X_val.values.astype(np.float32) if isinstance(X_val, pd.DataFrame) else X_val.astype(np.float32)
y_val_np = y_val.values.astype(int) if isinstance(y_val, pd.Series) else y_val.astype(int)

X_test_np = X_test.values.astype(np.float32) if isinstance(X_test, pd.DataFrame) else X_test.astype(np.float32)
y_test_np = y_test.values.astype(int) if isinstance(y_test, pd.Series) else y_test.astype(int)

In [ ]:
def objective(trial):
    """
    Optuna will run this function multiple times.
    In each trial, it picks a smart combination of parameters to test.
    """

    # Define the Hyperparameter Search Space
    n_d = trial.suggest_int('n_d', 8, 64, step=8)
    n_a = n_d # It is highly recommended to keep n_d and n_a equal
    n_steps = trial.suggest_int('n_steps', 3, 8)
    gamma = trial.suggest_float('gamma', 1.0, 2.0)
    lambda_sparse = trial.suggest_float('lambda_sparse', 1e-5, 1e-2, log=True)
    learning_rate = trial.suggest_float('lr', 1e-3, 1e-1, log=True)

    # Initialize TabNet with the suggested parameters
    clf = TabNetClassifier(
        n_d=n_d,
        n_a=n_a,
        n_steps=n_steps,
        gamma=gamma,
        lambda_sparse=lambda_sparse,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=learning_rate),
        scheduler_params={"step_size":10, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', # "entmax" yields better, sparser attention than "sparsemax"
        device_name='cuda',
        verbose=0
    )

    # Train the Model
    clf.fit(
        X_train=X_train_np, y_train=y_train_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_name=['valid'],
        eval_metric=['logloss'], # Optimize logloss internally
        max_epochs=50,           # Keep relatively short for tuning
        patience=10,             # Early stopping if validation doesn't improve
        batch_size=1024,
        virtual_batch_size=128,
        weights=smoothed_weight_dict                # Fixes the imbalanced age classes
    )

    # 4. Evaluate and return the score Optuna needs to maximize
    preds = clf.predict(X_val_np)
    f1 = f1_score(y_val_np, preds, average='macro')

    return f1

# RUN THE BAYESIAN SEARCH
print("\nStarting Optuna Bayesian Optimization for TabNet...")

# Create the study. We want to MAXIMIZE the f1_macro score.
study = optuna.create_study(direction='maximize')


study.optimize(objective, n_trials=50)

print("\nBest TabNet Parameters Found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# TRAIN THE FINAL TABNET MODEL
print("\nTraining Final TabNet Model with Best Parameters...")

# Extract the winning parameters
best_params = study.best_params
best_n_d = best_params['n_d']

final_tabnet = TabNetClassifier(
    n_d=best_n_d,
    n_a=best_n_d,
    n_steps=best_params['n_steps'],
    gamma=best_params['gamma'],
    lambda_sparse=best_params['lambda_sparse'],
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=best_params['lr']),
    mask_type='entmax',
    device_name='cuda',
    verbose=1
)

final_tabnet.fit(
    X_train=X_train_np, y_train=y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    eval_name=['valid'],
    eval_metric=['logloss'],
    max_epochs=150, # Let the final model train much longer
    patience=20,
    batch_size=1024, virtual_batch_size=128,
    weights=smoothed_weight_dict
)

[I 2026-04-28 00:05:00,646] A new study created in memory with name: no-name-1b3e576b-2012-411e-a070-800477491d92



Starting Optuna Bayesian Optimization for TabNet...
Stop training because you reached max_epochs = 50 with best_epoch = 48 and best_valid_logloss = 0.82628


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-28 00:28:58,736] Trial 0 finished with value: 0.5396124118506135 and parameters: {'n_d': 16, 'n_steps': 7, 'gamma': 1.4972166722329208, 'lambda_sparse': 0.000290089603426012, 'lr': 0.0026987040102355816}. Best is trial 0 with value: 0.5396124118506135.


Stop training because you reached max_epochs = 50 with best_epoch = 48 and best_valid_logloss = 0.34839


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-28 00:56:19,377] Trial 1 finished with value: 0.8390142514632228 and parameters: {'n_d': 56, 'n_steps': 8, 'gamma': 1.5353824464423238, 'lambda_sparse': 0.00012435826752850673, 'lr': 0.0582473706241945}. Best is trial 1 with value: 0.8390142514632228.


...

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-28 13:55:03,154] Trial 48 finished with value: 0.8561793908124239 and parameters: {'n_d': 56, 'n_steps': 4, 'gamma': 1.4539179246823752, 'lambda_sparse': 0.007254148715676199, 'lr': 0.057313888827958334}. Best is trial 19 with value: 0.8958749853239427.


Stop training because you reached max_epochs = 50 with best_epoch = 48 and best_valid_logloss = 0.2311


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-04-28 14:08:09,159] Trial 49 finished with value: 0.9054586119148512 and parameters: {'n_d': 64, 'n_steps': 3, 'gamma': 1.4008026020314386, 'lambda_sparse': 4.798433326657619e-05, 'lr': 0.009842269426968387}. Best is trial 49 with value: 0.9054586119148512.
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")



Best TabNet Parameters Found:
  n_d: 64
  n_steps: 3
  gamma: 1.4008026020314386
  lambda_sparse: 4.798433326657619e-05
  lr: 0.009842269426968387

Training Final TabNet Model with Best Parameters...
epoch 0  | loss: 1.11946 | valid_logloss: 1.00387 |  0:00:13s
epoch 1  | loss: 0.94788 | valid_logloss: 0.85243 |  0:00:27s
epoch 2  | loss: 0.8927  | valid_logloss: 0.81593 |  0:00:41s
epoch 3  | loss: 0.84466 | valid_logloss: 0.77692 |  0:00:55s
epoch 4  | loss: 0.79809 | valid_logloss: 0.74838 |  0:01:09s
epoch 5  | loss: 0.75257 | valid_logloss: 0.71351 |  0:01:23s
epoch 6  | loss: 0.7102  | valid_logloss: 0.68141 |  0:01:37s
epoch 7  | loss: 0.66528 | valid_logloss: 0.65409 |  0:01:51s
epoch 8  | loss: 0.62145 | valid_logloss: 0.62757 |  0:02:05s
epoch 9  | loss: 0.5814  | valid_logloss: 0.60874 |  0:02:19s
epoch 10 | loss: 0.54015 | valid_logloss: 0.57149 |  0:02:32s
epoch 11 | loss: 0.50413 | valid_logloss: 0.53576 |  0:02:46s
epoch 12 | loss: 0.47113 | valid_logloss: 0.52535 |  0:

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


**Best Params**

In [ ]:
# Best TabNet Parameters Found:
#   n_d: 64
#   n_steps: 3
#   gamma: 1.4008026020314386
#   lambda_sparse: 4.798433326657619e-05
#   lr: 0.009842269426968387